#### Importing required libraries 

In [11]:
import pandas as pd
from sklearn.metrics import *
from utils import Hetero_Data_Processor_Transfer_Learning
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from torch_geometric.nn import HANConv, Linear
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import torch
from torch.nn.functional import cross_entropy
from sklearn.preprocessing import RobustScaler
from torch_geometric.data import HeteroData
import torch_geometric.transforms as T
import torch.nn.functional as F
from torch import nn
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")

#### Testing a single load 

In [2]:
train_dataset = 'charlie_hebdo'
test_dataset = 'ottawashooting'
time_cut =60*3*24
processor = Hetero_Data_Processor_Transfer_Learning(train_dataset, test_dataset, time_cut=time_cut,test_size=0.3)
data = processor.process()

rumour
1    139
0    119
Name: count, dtype: int64


In [3]:
data

HeteroData(
  id={
    x=[2859, 106],
    y=[2859],
    train_mask=[2859],
    val_mask=[2859],
    test_mask=[2859],
  },
  reply_user_id={ x=[26437, 104] },
  (id, retweet, reply_user_id)={ edge_index=[2, 26437] },
  (reply_user_id, rev_retweet, id)={ edge_index=[2, 26437] }
)

In [6]:
class HAN(nn.Module):

    """
    Heterogeneous Graph Attention Network (HAN) model with two HANConv layers.

    This model is designed for heterogeneous graphs where nodes and edges may
    have different types. It uses hierarchical attention mechanisms to learn
    node representations by aggregating semantic information from multiple
    relations. After two stages of relational attention, the model outputs
    predictions for the `'id'` node type.

    Parameters
    ----------
    dim_in : int
        Input feature dimension shared across node types.
    dim_out : int
        Output feature dimension, typically the number of prediction classes.
    dim_h : int, optional (default=64)
        Hidden dimension used in each HANConv layer.
    heads : int, optional (default=4)
        Number of attention heads in each HANConv layer.

    Attributes
    ----------
    han : HANConv
        First hierarchical attention convolution layer.
    han2 : HANConv
        Second hierarchical attention convolution layer for deeper semantic aggregation.
    linear : nn.Linear
        Final linear projection applied to `'id'` node embeddings.

    Forward Inputs
    --------------
    x_dict : dict[str, torch.Tensor]
        Dictionary mapping node types to feature matrices.
    edge_index_dict : dict[str, torch.Tensor]
        Dictionary mapping edge types to adjacency information.

    Returns
    -------
    torch.Tensor
        Output predictions/logits for `'id'` nodes with shape
        [num_id_nodes, dim_out].
    """
    
    def __init__(self, dim_in, dim_out, dim_h=64, heads=4):
        super().__init__()
        self.han = HANConv(dim_in, dim_h, heads=heads,dropout=0.2, metadata=data.metadata())
        self.han2 = HANConv(dim_h, dim_h, heads=heads, dropout=0.2, metadata=data.metadata())
        self.linear = nn.Linear(dim_h, dim_out)

    def forward(self, x_dict, edge_index_dict):
        out = self.han(x_dict, edge_index_dict)
        out = self.han2(out, edge_index_dict)
        out = self.linear(out['id'])
        return out
    

In [7]:
def evaluate(model, data, mask_names):

    """
    Evaluate a graph classification model using masked subsets of the data.

    This function runs the model in evaluation mode, computes predictions,
    filters them using one or multiple masks from the input dataset, and
    returns common binary classification metrics.

    Parameters
    ----------
    model : torch.nn.Module
        Trained GNN model producing class logits from graph inputs.
    data : torch_geometric.data.HeteroData
        Heterogeneous graph data structure containing:
        - `x_dict`: dictionary of node feature matrices
        - `edge_index_dict`: dictionary of edge connectivity
        - `'id'` node type with attributes `y` and boolean masks
          (e.g., 'train_mask', 'val_mask', 'test_mask')
    mask_names : str or list[str]
        Name(s) of mask attributes to evaluate on. If multiple masks
        are provided, they are combined using logical OR.

    Returns
    -------
    tuple(float, float, float, float)
        A tuple containing:
        - acc : float
            Accuracy score.
        - precision : float
            Proportion of predicted positives that are correctly classified.
        - recall : float
            True positive rate.
        - auc : float
            ROC-AUC score based on predicted class probabilities.

    Notes
    -----
    - Metrics are computed only on masked nodes.
    - If ROC-AUC cannot be computed due to a single class present in labels,
      a value of 0.0 is returned.
    """


    model.eval()
    out = model(data.x_dict, data.edge_index_dict)
    preds = out.argmax(dim=-1)
    labels = data['id'].y

    if isinstance(mask_names, str):
        mask = data['id'][mask_names]
    else:
        mask = torch.zeros_like(data['id'].y, dtype=torch.bool)
        for name in mask_names:
            mask |= data['id'][name]

    preds_masked = preds[mask]
    labels_masked = labels[mask]
    probs = out[mask][:, 1]  # Fixed: out is a tensor

    acc = accuracy_score(labels_masked.cpu(), preds_masked.cpu())
    precision = precision_score(labels_masked.cpu(), preds_masked.cpu(), zero_division=0)
    recall = recall_score(labels_masked.cpu(), preds_masked.cpu(), zero_division=0)

    try:
        auc = roc_auc_score(labels_masked.cpu(), probs.detach().cpu())
    except ValueError:
        auc = 0.0

    return acc, precision, recall, auc

In [8]:
def train(model, data, optimizer, epochs=100):

    """
    Train a graph neural network on masked node labels and monitor performance.

    This function performs a full training loop where the loss is computed only
    over nodes marked by the `'train_mask'` attribute in the heterogeneous
    graph's `'id'` node type. At each epoch, training metrics are logged, and
    validation metrics are evaluated periodically to track generalization.

    Parameters
    ----------
    model : torch.nn.Module
        The GNN model to be trained. Must accept heterogeneous node features
        (`x_dict`) and edge connectivity (`edge_index_dict`) in its forward pass.
    data : torch_geometric.data.HeteroData
        A heterogeneous graph containing:
        - `x_dict`: node feature dictionaries
        - `edge_index_dict`: adjacency per edge type
        - `'id'` node labels stored in `.y`
        - masks such as `'train_mask'`, `'val_mask'`, `'test_mask'`
    optimizer : torch.optim.Optimizer
        The optimizer used to update model weights.
    epochs : int, optional (default=100)
        Number of training epochs.

    Notes
    -----
    - Loss is computed using cross-entropy over masked nodes.
    - Metrics include accuracy, precision, recall, and ROC-AUC.
    - Validation performance is printed every 10 epochs.
    - At the end of training, results are evaluated on a combined
      validation + test mask for a final performance estimate.

    Returns
    -------
    None
        This function prints training and evaluation logs but does not return a value.
    """

    
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()

        out = model(data.x_dict, data.edge_index_dict)
        mask = data['id'].train_mask
        #out_id = out['id']
        loss = F.cross_entropy(out[mask], data['id'].y[mask])
        #loss = cross_entropy(out_id[data['id'].train_mask], data['id'].y[data['id'].train_mask])
        loss.backward()
        optimizer.step()

        # Train metrics
        acc, precision, recall, auc = evaluate(model, data, 'train_mask')
        print(f"[Epoch {epoch:03d}] Train - Acc: {acc:.4f} | Prec: {precision:.4f} | Recall: {recall:.4f} | AUC: {auc:.4f}")

        # Val metrics every 10 epochs
        if epoch % 10 == 0:
            acc_val, prec_val, recall_val, auc_val = evaluate(model, data, 'val_mask')
            print(f"[Epoch {epoch:03d}] Val   - Acc: {acc_val:.4f} | Prec: {prec_val:.4f} | Recall: {recall_val:.4f} | AUC: {auc_val:.4f}")

    print("\nFinal Evaluation (Val + Test):")
    acc_final, prec_final, recall_final, auc_final = evaluate(model, data, ['val_mask', 'test_mask'])
    print(f"[Final] Val+Test - Acc: {acc_final:.4f} | Prec: {prec_final:.4f} | Recall: {recall_final:.4f} | AUC: {auc_final:.4f}")




#### Example  training

In [12]:

model = HAN(dim_in=-1, dim_out=2)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data, model = data.to(device), model.to(device)

In [13]:
train(model, data, optimizer, epochs=100)


[Epoch 001] Train - Acc: 0.7059 | Prec: 0.0000 | Recall: 0.0000 | AUC: 0.6790
[Epoch 002] Train - Acc: 0.7059 | Prec: 0.0000 | Recall: 0.0000 | AUC: 0.6786
[Epoch 003] Train - Acc: 0.7059 | Prec: 0.0000 | Recall: 0.0000 | AUC: 0.6772
[Epoch 004] Train - Acc: 0.7059 | Prec: 0.0000 | Recall: 0.0000 | AUC: 0.6739
[Epoch 005] Train - Acc: 0.7059 | Prec: 0.0000 | Recall: 0.0000 | AUC: 0.6713
[Epoch 006] Train - Acc: 0.7059 | Prec: 0.0000 | Recall: 0.0000 | AUC: 0.6705
[Epoch 007] Train - Acc: 0.7059 | Prec: 0.0000 | Recall: 0.0000 | AUC: 0.6725
[Epoch 008] Train - Acc: 0.7059 | Prec: 0.0000 | Recall: 0.0000 | AUC: 0.6772
[Epoch 009] Train - Acc: 0.7059 | Prec: 0.0000 | Recall: 0.0000 | AUC: 0.6846
[Epoch 010] Train - Acc: 0.7059 | Prec: 0.0000 | Recall: 0.0000 | AUC: 0.6943
[Epoch 010] Val   - Acc: 0.5116 | Prec: 0.0000 | Recall: 0.0000 | AUC: 0.6176
[Epoch 011] Train - Acc: 0.7059 | Prec: 0.0000 | Recall: 0.0000 | AUC: 0.7059
[Epoch 012] Train - Acc: 0.7059 | Prec: 0.0000 | Recall: 0.0000 

#### Setting MLflow Experiment

In [14]:
mlflow.set_experiment("Han Network 2025-10-19 Ferguson TF")

<Experiment: artifact_location='/workspaces/rumour-detection-gnn/New experiments/mlruns/52', creation_time=1761008283150, experiment_id='52', last_update_time=1761008283150, lifecycle_stage='active', name='Han Network 2025-10-19 Ferguson TF', tags={}>

#### Loading dataset statistics to get the final time cut 

In [18]:
df_posts_by_time_cut = pd.read_csv('ottawa_shooting_posts_by_time_cut.csv')


In [19]:
time_cut_last_post = int(df_posts_by_time_cut[df_posts_by_time_cut.post==\
                         int(df_posts_by_time_cut['post'].max())].time_cut.min())

In [20]:
time_cut_last_post

599

**Creating evaluate_metrics function to assess classification when new posts are created**

In [ ]:
def evaluate_metrics(model, data, mask):

    """
    Evaluate classification performance of a trained model on masked node subsets.

    This function computes predictions using the model in evaluation mode,
    extracts only the nodes specified by the provided mask, and calculates
    multiple binary classification metrics using both predicted classes and
    predicted probabilities.

    Parameters
    ----------
    model : torch.nn.Module
        A trained graph neural network model that outputs logits for the `'id'` node type.
    data : torch_geometric.data.HeteroData
        Heterogeneous graph data containing node features (`x_dict`),
        edge indices (`edge_index_dict`), ground truth labels (`y`),
        and boolean evaluation masks for `'id'` nodes.
    mask : torch.Tensor or list[bool]
        Boolean mask that selects which `'id'` nodes to include in metric computation.

    Returns
    -------
    tuple(float, float, float, float)
        - acc : float
            Accuracy score for masked nodes.
        - prec : float
            Macro-averaged precision.
        - recall : float
            Macro-averaged recall.
        - auc : float
            ROC-AUC score computed using probability of the positive class.

    Notes
    -----
    - Evaluation is performed within a `torch.no_grad()` block to avoid
      gradient tracking during inference.
    - If AUC computation fails due to only one class present in the mask,
      a fallback value of `0.0` is returned.
    """
    
    model.eval()
    with torch.no_grad():
        out = model(data.x_dict, data.edge_index_dict)
        preds = out.argmax(dim=1)
        probs = out[:, 1]  # Probability of class 1

    true = data['id'].y[mask]
    pred = preds[mask]
    prob = probs[mask]

    acc = accuracy_score(true.cpu(), pred.cpu())
    prec = precision_score(true.cpu(), pred.cpu(), average='macro', zero_division=0)
    recall = recall_score(true.cpu(), pred.cpu(), average='macro', zero_division=0)
    try:
        auc = roc_auc_score(true.cpu(), prob.cpu())
    except:
        auc = 0.0

    return acc, prec, recall, auc

* **The initial  time cut will be 10 minutes after the first post publication**
*  **The final time cut will be equal to 6 hours after the publication of last post**

In [10]:

    
previous_node_count = 0  # Start with no nodes

for time_cut in range(10, max_time_cut+(60*6), 10):
    
    print(f"\n=== Time Cut: {time_cut} ===")
    train_dataset = 'charlie_hebdo'
    #test_dataset = 'ferguson'
    #test_dataset = 'sydneysiege'
    #test_dataset = 'germanwings_crash'
    test_dataset = 'ottawashooting'
    time_cut =time_cut
    processor = Hetero_Data_Processor_Transfer_Learning(train_dataset, test_dataset, time_cut=time_cut,test_size=0.3)
    data = processor.process()


    model = HAN(dim_in=-1, dim_out=2)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    data, model = data.to(device), model.to(device)

    current_node_count = data['id'].x.shape[0]
    new_node_indices = np.arange(previous_node_count, current_node_count)
    previous_node_count = current_node_count

    # Compute imbalance
    y_train = data['id'].y[data['id'].train_mask].cpu()
    imbalance = (y_train == 1).sum() / len(y_train)

    with mlflow.start_run(run_name=f"time_cut_{time_cut}"):
        for epoch in range(1, 101):
            model.train()
            optimizer.zero_grad()
            out = model(data.x_dict, data.edge_index_dict)
            mask = data['id'].train_mask
            loss = F.cross_entropy(out[mask], data['id'].y[mask])
            loss.backward()
            optimizer.step()

            if epoch % 100 == 0:
                 train_acc, train_prec, train_recall, train_auc= evaluate_metrics(model, data, data['id'].train_mask)
                 print(f"[Epoch {epoch}] Train Loss: {loss:.4f} | Train Recall: {train_recall:.4f} | Train Auc: {train_auc:.4f}")
        # Evaluate all predictions
        model.eval()
        with torch.no_grad():
            out = model(data.x_dict, data.edge_index_dict)
            preds = out.argmax(dim=1)
            probs = out[:, 1]
    
        # New instances in val/test set
        val_test_mask = (data['id'].val_mask | data['id'].test_mask).cpu().numpy()
        new_instance_mask = np.zeros_like(val_test_mask, dtype=bool)
        new_instance_mask[new_node_indices] = True
        final_mask = new_instance_mask & val_test_mask
    
        if final_mask.sum() > 0:
            # Compute metrics
            true_new = data['id'].y.cpu().numpy()[final_mask]
            pred_new = preds.cpu().numpy()[final_mask]
            prob_new = probs.cpu().numpy()[final_mask]
        
            new_precision = precision_score(true_new, pred_new, average='macro', zero_division=0)
            new_recall = recall_score(true_new, pred_new, average='macro', zero_division=0)
            new_acc = accuracy_score(true_new, pred_new)
        else:
            new_precision = 0
            new_recall = 0
            new_acc =0
            print("No new instances to evaluate.")
    
    
        all_eval_mask = data['id'].val_mask | data['id'].test_mask
        acc, prec, recall, auc = evaluate_metrics(model, data, all_eval_mask)
        
        print(f"[Final Val+Test] Acc: {acc:.4f} | Prec: {prec:.4f} | Recall: {recall:.4f} | AUC: {auc:.4f}")
    
    
        print(f"New Instances: {final_mask.sum()}")
        print(f"New Precision: {new_precision:.4f} | New Recall: {new_recall:.4f}")

        mlflow.log_metric("new_posts", final_mask.sum())
        
        mlflow.log_metric("final_precision", prec)
        mlflow.log_metric("final_recall", recall)
        mlflow.log_metric("final_auc", auc)
        mlflow.log_metric("final_acc", acc)

        mlflow.log_metric("curr_precision", new_precision)
        mlflow.log_metric("curr_recall", new_recall)
        mlflow.log_metric("curr_acc", new_acc)

        mlflow.log_metric("time_cut", time_cut)



=== Time Cut: 10 ===
rumour
0    5
1    1
Name: count, dtype: int64
[Epoch 100] Train Loss: 0.3390 | Train Recall: 0.8253 | Train Auc: 0.9161
[Final Val+Test] Acc: 0.6667 | Prec: 0.4000 | Recall: 0.4000 | AUC: 0.2000
New Instances: 6
New Precision: 0.4000 | New Recall: 0.4000

=== Time Cut: 20 ===
rumour
0    12
1     2
Name: count, dtype: int64
[Epoch 100] Train Loss: 0.3482 | Train Recall: 0.8159 | Train Auc: 0.9014
[Final Val+Test] Acc: 0.9286 | Prec: 0.9615 | Recall: 0.7500 | AUC: 0.8333
New Instances: 8
New Precision: 1.0000 | New Recall: 1.0000

=== Time Cut: 30 ===
rumour
0    20
1     2
Name: count, dtype: int64
[Epoch 100] Train Loss: 0.3345 | Train Recall: 0.8233 | Train Auc: 0.9143
[Final Val+Test] Acc: 0.8636 | Prec: 0.6404 | Recall: 0.7000 | AUC: 0.8250
New Instances: 8
New Precision: 0.5000 | New Recall: 0.4375

=== Time Cut: 40 ===
rumour
0    26
1     3
Name: count, dtype: int64
[Epoch 100] Train Loss: 0.3480 | Train Recall: 0.8136 | Train Auc: 0.9060
[Final Val+Test] 